In [1]:
# 战备
import os
import sys
from pathlib import Path

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

WORKSPACE_ROOT = Path("/vla/workspace/my_tbot")
SRC_ROOT = WORKSPACE_ROOT / "src"
MODELS_ROOT = Path("/vla/.models")
DATA_ROOT = Path("/vla/workspace/data")

### 1. 模型加载

In [2]:
from lerobot.configs.policies import PreTrainedConfig
MODEL_ID = Path("/vla/workspace/models/bp_tbot_init")
device ='cuda'
QWEN3_VL_PATH = Path("/vla/workspace/models/Qwen3-VL-2B-Instruct")
COSMOS_PATH = Path("/vla/workspace/models/Cosmos-Tokenizer-CI8x8")
DA3_PATH = Path("/vla/workspace/models/DA3-LARGE-1.1")
DA3_CODE_ROOT = Path("/vla/workspace/my_tbot/third_party/Depth-Anything-3")

from lerobot.policies.BP_TBot.configuration_bp_tbot import BPTBotConfig
bp_policy_cfg = BPTBotConfig(
    device=None,
    chunk_size=50,
    n_action_steps=50,
    n_obs_steps=1,
    max_state_dim=32,
    max_action_dim=32,
    image_delta_indices=[-15, 0, 15],
    bp_num_chunks=4,
    bp_action_chunk_size=50,
)

bp_policy_cfg.device = device
bp_policy_cfg.pretrained_path = MODEL_ID
bp_policy_cfg.qwen3_vl_variant = "qwen3_vl_28l"
bp_policy_cfg.action_expert_variant = "qwen3_28l"
bp_policy_cfg.qwen3_vl_pretrained_path = str(QWEN3_VL_PATH)
bp_policy_cfg.cosmos_tokenizer_path_or_name = str(COSMOS_PATH)
bp_policy_cfg.enable_3d_queries = True
bp_policy_cfg.num_3d_query_tokens = 432
bp_policy_cfg.lambda_3d = 0.01
bp_policy_cfg.da3_model_path_or_name = str(DA3_PATH)
bp_policy_cfg.da3_code_root = str(DA3_CODE_ROOT)
bp_policy_cfg.log_da3_teacher_timing = True
bp_policy_cfg.bp_num_chunks = 4
bp_policy_cfg.bp_action_chunk_size = 50
bp_policy_cfg.bp_use_type_embedding = True
bp_policy_cfg.bp_use_chunk_embedding = True
bp_policy_cfg.bp_use_action_step_embedding = True
bp_policy_cfg.validate_features() 
# 19.8s

/vla/.conda/miniconda3/envs/mytbot/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
from lerobot.policies.BP_TBot.modeling_bp_tbot import BPTBotPolicy

policy = BPTBotPolicy.from_pretrained(MODEL_ID,config=bp_policy_cfg).to(device).eval()
print(policy.model.bp_action_mlp)
print(policy.model.bp_state_mlp)
print(policy.model.bp_chunk_embedding)
# 1m47.5s

[WARN ] Dependency `gsplat` is required for rendering 3DGS. Install via: pip install git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70
[INFO ] using MLP layer as FFN
Loading weights from local directory
Loading weights from local directory
Sequential(
  (0): LayerNorm((1600,), eps=1e-05, elementwise_affine=True)
  (1): Linear(in_features=1600, out_features=4096, bias=True)
  (2): SiLU()
  (3): Linear(in_features=4096, out_features=4096, bias=True)
  (4): SiLU()
  (5): Linear(in_features=4096, out_features=2048, bias=True)
)
Sequential(
  (0): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
  (1): Linear(in_features=32, out_features=2048, bias=True)
  (2): SiLU()
  (3): Linear(in_features=2048, out_features=2048, bias=True)
)
Embedding(4, 2048)


## 2. 数据集加载

In [4]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset
# 必须对齐训练时的代码，即需要有timestamps

delta_timestamps = {
    "action": [i / 30 for i in range(50)],
    "observation.images.cam_high": [-0.5, 0.0, 0.5],
    "observation.images.cam_left_wrist": [-0.5, 0.0, 0.5],
    "observation.images.cam_right_wrist": [-0.5, 0.0, 0.5],
}

da_current = LeRobotDataset('/vla/workspace/data/adjust_bottle/aloha-agilex_clean_50',delta_timestamps=delta_timestamps)
da_prompt = LeRobotDataset('/vla/workspace/data/adjust_bottle/aloha-agilex_clean_50')
from lerobot.datasets.behavior_prompt_dataset import BehaviorPromptLeRobotDataset,BehaviorPromptConfig
config = BehaviorPromptConfig(prompt_action_chunk_size=50, 
    max_prompt_chunks=None, same_episode_policy='avoid', 
    seed=0, num_chunks=4, height=224, width=224, max_state_dim=32, max_action_dim=32, 
    qwen3_vl_processor_path='Qwen/Qwen3-VL-2B-Instruct', action_mode='delta'
)

bp_ds = BehaviorPromptLeRobotDataset.with_default_transforms(da_current, da_prompt, config)

Hydrating transform InjectMissingStateActionTransformFn (robot_type=aloha, resolved=aloha, action_seq_len=1, state_seq_len=1, placeholder_dim=14)
Hydrating transform NormalizeTransformFn with dataset.meta.stats (robot_type=aloha, resolved=aloha) and selected_keys (selected_keys=['observation.state', 'action'])
Hydrating transform ComposeFieldsTransform with mapping (robot_type=aloha, resolved=aloha)
Hydrating transform DeltaActionTransformFn with mapping and mask (robot_type=aloha, resolved=aloha)
Hydrating transform RemapImageKeyTransformFn with mapping (robot_type=aloha, resolved=aloha)


In [5]:
for i, step in enumerate(bp_ds.transform.transforms):
    print(f"data process step:  [{i}] {step.__class__.__name__}")

data process step:  [0] BPPadOrSampleChunksFn
data process step:  [1] BPResizeImagesWithPadFn
data process step:  [2] BPRemapImageKeyTransformFn
data process step:  [3] BPNormalizeTransformFn
data process step:  [4] BPComposeFieldsTransform
data process step:  [5] BPDeltaActionTransformFn
data process step:  [6] BPPadStateAndActionTransformFn
data process step:  [7] BPImgOnlyQwen3VLTransformFn
data process step:  [8] InjectMissingStateActionTransformFn
data process step:  [9] DeltaActionTransformFn
data process step:  [10] ResizeImagesWithPadFn
data process step:  [11] RemapImageKeyTransformFn
data process step:  [12] NormalizeTransformFn
data process step:  [13] ComposeFieldsTransform
data process step:  [14] PadStateAndActionTransformFn
data process step:  [15] ImgOnlyQwen3VLTransformFn
data process step:  [16] UnifyBPInputsTransformFn


## 3. 推理与训练

In [6]:
from torch.utils.data import DataLoader
from torch.utils.data._utils.collate import default_collate
# 用bp_ds[0],bp_ds[1] 构建batch size =2 的batch
samples = [bp_ds[0],bp_ds[1]]
batch = default_collate(samples)

In [7]:
import torch
def move_to_device(x, device):
    if isinstance(x, torch.Tensor):
        return x.to(device)
    if isinstance(x, dict):
        return {k: move_to_device(v, device) for k, v in x.items()}
    if isinstance(x, list):
        return [move_to_device(v, device) for v in x]
    if isinstance(x, tuple):
        return tuple(move_to_device(v, device) for v in x)
    return x
batch_for_forward = move_to_device(batch, device)



### 训练

In [8]:
import torch
policy.train()
with torch.no_grad():
    loss, loss_dict_reloaded = policy.forward(batch_for_forward)
loss

[INFO ] Selecting reference view using strategy: saddle_balanced


tensor(0.3502, device='cuda:0')

### 推理

In [9]:
policy.eval()
actions, _ = policy.predict_action_chunk(batch_for_forward)
actions


tensor([[[ 1.2269e-01, -1.6003e-01, -3.0353e-02,  ...,  2.8968e-03,
          -3.4248e-03, -2.0657e-03],
         [ 1.6048e-01, -1.8825e-01, -9.9670e-02,  ..., -5.6552e-03,
          -1.8913e-02,  3.3836e-03],
         [ 2.1158e-01, -2.2403e-01, -7.7022e-02,  ..., -3.7030e-03,
           7.3399e-04,  1.4818e-02],
         ...,
         [-1.8547e-01, -7.3538e-01,  3.1839e-01,  ..., -5.7675e-03,
          -2.1195e-02,  1.7832e-02],
         [-2.3333e-01, -7.5026e-01,  3.7058e-01,  ..., -8.5841e-03,
          -3.0550e-02,  2.9771e-02],
         [-2.2887e-01, -7.3980e-01,  3.4724e-01,  ..., -3.0906e-04,
          -2.3205e-02,  1.8033e-02]],

        [[ 6.1060e-02, -7.0092e-02, -3.8198e-02,  ..., -4.6122e-03,
           1.4908e-02,  1.0896e-02],
         [ 4.4197e-02, -8.4330e-02, -1.7004e-02,  ..., -5.7766e-03,
           1.0789e-02,  8.4754e-03],
         [ 8.6086e-02, -7.1532e-02, -1.6873e-02,  ...,  1.3496e-03,
           8.8788e-03,  2.9687e-02],
         ...,
         [-5.1053e-01,  6